# CineData Analytics — Agente Text-to-SQL (demo)

Este notebook demonstra o agente respondendo perguntas em linguagem natural sobre o catálogo de
filmes (`cinerocket.db`), usando um modelo gratuito via **OpenRouter** com *tool calling*.

**Antes de rodar:**
1. Copie `.env.example` para `.env` na raiz do projeto e preencha `OPENROUTER_API_KEY`
   (gerada em https://openrouter.ai/keys).
2. Copie o arquivo `cinerocket.db` fornecido pelo professor para a pasta `data/`.
3. Instale as dependências: `pip install -r requirements.txt`.

> ⚠️ **Cada pergunta consome de 2 a 4 requisições da cota diária do OpenRouter (50/dia no free
> tier).** Este notebook faz poucas perguntas de propósito — não rode a mesma célula várias vezes
> sem necessidade (o agente já tem cache de respostas, ver `app/cache.py`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))

from app.config import settings
print("Banco:", settings.database_path, "existe?", settings.database_path.exists())
print("Modelos configurados (ordem de fallback):", settings.models)

## 1. Conferência rápida do schema
Mostra quantas linhas cada tabela tem — útil para confirmar que o banco carregou certo antes de
gastar qualquer chamada de API.

In [ ]:
from app.schema import resumo_schema_legivel
print(resumo_schema_legivel())

## 2. Instanciando o agente

O agente (`app/agent.py`) é um loop de *tool calling* escrito à mão sobre o SDK `openai` apontado
para o OpenRouter. Ele recebe o schema completo do banco e uma lista de armadilhas de dados
conhecidas (ver `app/schema.py`) já no prompt do sistema — isso evita chamadas extras só para o
modelo "descobrir" o schema, economizando cota.

In [ ]:
from app.agent import CinerocketAgent

agente = CinerocketAgent()

## 3. Perguntas de exemplo — uma por categoria do enunciado

Cada célula abaixo faz **uma pergunta real** (gasta de 2 a 4 chamadas de API). Rode uma de cada
vez e observe: a resposta em português, o SQL que o modelo gerou sozinho, e quantas chamadas
foram usadas.

In [ ]:
def perguntar_e_mostrar(pergunta: str):
    resultado = agente.perguntar(pergunta)
    print("PERGUNTA:", resultado.pergunta)
    print("MODELO:", resultado.modelo_usado, "| CHAMADAS API:", resultado.chamadas_api, "| CACHE:", resultado.do_cache)
    print("\nSQL EXECUTADO:")
    for sql in resultado.consultas_sql:
        print(" -", " ".join(sql.split()))
    print("\nRESPOSTA:\n", resultado.resposta)
    return resultado

In [ ]:
# Bilheteria e Finanças
r1 = perguntar_e_mostrar("Quais são os 10 filmes com maior receita em reais?")

In [ ]:
# Popularidade e Engajamento
r2 = perguntar_e_mostrar("Quais são os 5 filmes mais populares?")

In [ ]:
# Elenco e Equipe
r3 = perguntar_e_mostrar("Qual ator teve mais participações em filmes lançados nos últimos 5 anos?")

In [ ]:
# Gêneros e Produtoras — pergunta que exige aplicar a regra de negócio "receita informada" sem que o usuário peça explicitamente
r4 = perguntar_e_mostrar("Qual produtora teve o maior lucro total?")

In [ ]:
# Avaliações dos Usuários
r5 = perguntar_e_mostrar("Quais filmes têm a nota média dos usuários mais divergente da nota do IMDb?")

## 4. Bônus: agente híbrido (SQL + busca textual sobre sinopses)

Esta pergunta não tem uma coluna estruturada para filtrar — o agente deve perceber isso e usar a
ferramenta `buscar_sinopses_similares` (TF-IDF sobre a sinopse) em vez de tentar escrever um SQL
com `LIKE`.

In [ ]:
r6 = perguntar_e_mostrar("Me indique filmes com uma história parecida com viagem no tempo para impedir uma invasão alienígena")

## 5. Rodando a mesma pergunta de novo (deve vir do cache, sem gastar cota)

In [ ]:
r1_repetido = agente.perguntar("Quais são os 10 filmes com maior receita em reais?")
print("Veio do cache?", r1_repetido.do_cache, "| chamadas de API nesta repetição:", r1_repetido.chamadas_api)

## 6. Avaliação com gabarito (dry-run, sem gastar cota)

`eval/golden_questions.py` tem 14 perguntas com SQL de referência escrito à mão, cobrindo as 5
categorias do enunciado. O modo `--dry-run` só mostra o gabarito; para comparar com o agente de
verdade (gastando cota), use `python -m eval.run_eval --limit N` pelo terminal.

In [ ]:
!cd .. && python -m eval.run_eval --dry-run

## 7. Encerrando

In [ ]:
agente.close()